In [1]:
import pandas as pd
import numpy as np

In [2]:
# Loading the dataset
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [3]:
df.shape

(5000, 9)

In [4]:
df.columns

Index(['lead_source', 'industry', 'employment_status', 'location',
       'annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score', 'converted'],
      dtype='object')

In [5]:
df.dtypes

lead_source                  object
industry                     object
employment_status            object
location                     object
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [6]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [7]:
# Calculate the percentage of missing values in each column
df.isnull().mean() * 100

lead_source                 2.96
industry                    4.80
employment_status           3.88
location                    4.16
annual_income               7.38
number_of_courses_viewed    0.00
interaction_count           0.00
lead_score                  0.70
converted                   0.00
dtype: float64

In [8]:
# Categorical columns
categorical = ['lead_source','industry','employment_status','location']

# Numerical columns
numerical = ['annual_income','number_of_courses_viewed','interaction_count','lead_score']

In [9]:
# Target variable
target = 'converted'

In [10]:
# Replace missing values in categorical columns with 'NA'
df[categorical] = df[categorical].fillna('NA')

# Replace missing values in numerical columns with 0.0
df[numerical] = df[numerical].fillna(0.0)

# Check that there are no missing values left
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

## Question 1
What is the most frequent observation (mode) for the column industry?

In [11]:
# Find the most frequent value in the industry column
df['industry'].mode()

0    technology
Name: industry, dtype: object

## Question 2
Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

- interaction_count and lead_score
- number_of_courses_viewed and lead_score
- number_of_courses_viewed and interaction_count
- annual_income and interaction_count

In [12]:
# Create the correlation matrix
df[numerical].corr().round(2)

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.00,0.16,0.12,0.23
number_of_courses_viewed,0.16,1.00,0.72,0.76
interaction_count,0.12,0.72,1.00,0.92
lead_score,0.23,0.76,0.92,1.00


In [13]:
from sklearn.model_selection import train_test_split

# First split: 80% training data, 20% test data
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

# Second split: take 25% of the 80% training data as validation
# This gives us 60% train, 20% validation, 20% test
df_train, df_val = train_test_split(df_full_train, test_size=0.25,random_state=42)

# Check the number of rows in each dataset
print("Train:", len(df_train))
print("Validation:", len(df_val))
print("Test:", len(df_test))

Train: 3000
Validation: 1000
Test: 1000


In [14]:
# Separate the target variable from the training data
y_train = df_train['converted']
y_val = df_val['converted']

# Remove the target from the feature data
X_train = df_train.drop('converted', axis=1)
X_val = df_val.drop('converted', axis=1)

## Question 3
- Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.
- Round the scores to 2 decimals using round(score, 2).
  
Which of these variables has the biggest mutual information score?

- industry
- location
- lead_source
- employment_status

In [15]:
# Categorical features
categorical = ['industry','location','lead_source','employment_status']

from sklearn.metrics import mutual_info_score

# Calculate mutual information for each categorical feature

for column in categorical:
    score = mutual_info_score(X_train[column], y_train)
    print(column, round(score, 2))

industry 0.0
location 0.0
lead_source 0.03
employment_status 0.02


Question 4
- Now let's train a logistic regression.
- Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
- Fit the model on the training dataset.
- 1. To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
- 2.  model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
- Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

In [16]:
from sklearn.feature_extraction import DictVectorizer

# Convert the training features into dictionaries
train_dict = X_train.to_dict(orient='records')

# Convert the validation features into dictionaries
val_dict = X_val.to_dict(orient='records')

# Create the one-hot encoder
dv = DictVectorizer(sparse=False)

# Fit the encoder using the training data
X_train_encoded = dv.fit_transform(train_dict)

# Transform the validation data using the same encoder
X_val_encoded = dv.transform(val_dict)

In [17]:
from sklearn.linear_model import LogisticRegression

# Create the logistic regression model
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

# Train the model using the training data
model.fit(X_train_encoded, y_train)

# Predict the validation data
y_pred = model.predict(X_val_encoded)

In [18]:
from sklearn.metrics import accuracy_score

# Calculate the accuracy on the validation dataset
accuracy = accuracy_score(y_val, y_pred)
print(round(accuracy, 2))

0.65


## Question 5
Let's find the least useful feature using the feature elimination technique.
- Train a model using the same features and parameters as in Q4 (without rounding).
- Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
- For each feature, calculate the difference between the original accuracy and the accuracy without the feature.

Which of following feature has the smallest difference?

- 'lead_source'
- 'number_of_courses_viewed'
- 'interaction_count'
Note: The difference doesn't have to be positive.

In [21]:
# Save the original validation accuracy from Q4
original_accuracy = accuracy

# Features we want to test removing
features_to_remove = ['lead_source', 'number_of_courses_viewed', 'interaction_count']

# Test each feature separately
for feature in features_to_remove:

    # Remove the feature from the training and validation data
    X_train_new = X_train.drop(feature, axis=1)
    X_val_new = X_val.drop(feature, axis=1)

    # Convert the data into dictionaries
    train_dict = X_train_new.to_dict(orient='records')
    val_dict = X_val_new.to_dict(orient='records')

    # Create a new encoder
    dv_new = DictVectorizer(sparse=False)

    # Fit the encoder on training data
    X_train_new = dv_new.fit_transform(train_dict)

    # Transform validation data using the same encoder
    X_val_new = dv_new.transform(val_dict)

    # Create the same Logistic Regression model
    model_new = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=2
    )

    # Train the model
    model_new.fit(X_train_new, y_train)

    # Make predictions
    y_pred_new = model_new.predict(X_val_new)

    # Calculate accuracy
    accuracy_new = accuracy_score(y_val, y_pred_new)

    # Calculate the difference from the original accuracy
    difference = original_accuracy - accuracy_new

    print(feature, "accuracy:", round(accuracy_new, 3), "difference:", round(difference, 3))

lead_source accuracy: 0.642 difference: 0.003
number_of_courses_viewed accuracy: 0.643 difference: 0.002
interaction_count accuracy: 0.601 difference: 0.044


## Question 6
- Now let's train a regularized logistic regression.
- Let's try the following values of the parameter C: [0.000001, 0.00001, 0.0001, 0.001].
- Train models using all the features as in Q4.
- Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

Which of these C leads to the best accuracy on the validation set?

- 0.000001
- 0.00001
- 0.0001
- 0.001
  
Note: If there are multiple options, select the smallest C.

In [20]:
# Different C values we need to test
C_values = [0.000001, 0.00001, 0.0001, 0.001]

# Test each C value
for c in C_values:

    # Create the Logistic Regression model
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)

    # Train the model using all features
    model.fit(X_train_encoded, y_train)

    # Make predictions on validation data
    y_pred = model.predict(X_val_encoded)

    # Calculate validation accuracy
    accuracy = accuracy_score(y_val, y_pred)

    # Print accuracy rounded to 3 decimal places
    print("C:", c, "Accuracy:", round(accuracy, 3))

C: 1e-06 Accuracy: 0.598
C: 1e-05 Accuracy: 0.598
C: 0.0001 Accuracy: 0.613
C: 0.001 Accuracy: 0.645
